In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

In [2]:
df_participants1 = pd.read_csv('../../../Data/processed/pilot2/group1_participant_metrics.zip', compression='zip', header=0)
df_participants2 = pd.read_csv('../../../Data/processed/pilot2/group2_participant_metrics.zip', compression='zip', header=0)
df_participants = pd.read_csv('../../../Data/processed/pilot2/all_participant_metrics.zip', compression='zip', header=0)
df_participants_follow_up = pd.read_csv('../../../Data/processed/pilot2/all_participant_metrics_followup.zip', compression='zip', header=0)
df_time = pd.read_csv('../../../Data/processed/pilot2/metrics_results_over_time.csv', header=0)
print(df_participants1.shape)
print(df_participants2.shape)
print(df_participants.shape)
print(df_participants_follow_up.shape)
print(df_time.shape)

(558, 13)
(554, 13)
(1112, 13)
(811, 13)
(764, 23)


In [4]:
order = ['control', 'intervention_2', 'intervention_3', 'intervention_4', 'intervention_5', 'intervention_1',]
names = ['Control', 'Textual', 'Visual', 'Gamified', 'Feedback', 'Knowledge']
# colors = ['#B22222', '#72AEE6', '#4682B4', '#2C1D6C', '#7A6FCD', '#9370DB']
colors = ['#B22222', '#72AEE6', '#1F4B82', '#5C2D91', '#9B4D96', '#D16DB3']
colors = ['#B22222', '#5C2D91', '#D16DB3', '#72AEE6', '#1F4B82', '#4D8E8A']

name_mapping = dict(zip(order, names))

def capped_error(mean, std, n, cap=100):
    std_error = std / np.sqrt(n)
    lower = std_error
    upper = min(std_error, cap - mean)
    return lower, upper

# Function to add significance brackets
def add_significance_bracket(ax, x1, x2, y, h, text):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='black')
    ax.text((x1+x2)*.5, y+h-1.2, text, ha='center', va='bottom', fontsize=20)

In [26]:
def plot_single_timepoint(df, file_name, metric, time_label, brackets=None, y_max=109, metric_label=None):
    fig, ax = plt.subplots(figsize=(5, 5))

    # Calculate positions for data points (same spacing as original)
    spacing = 0.15
    positions = np.arange(len(order)) * spacing

    means = []

    for i, group in enumerate(order):
        column = f"{metric}"
        group_data = df[df['intervention_group'] == group][column]

        # Calculate the KDE
        kde = gaussian_kde(group_data)
        x_range = np.linspace(group_data.min(), group_data.max(), 100)
        kde_values = kde(x_range)

        # Scale the KDE values (same scaling as original)
        scaling_factor = 0.07
        kde_values = kde_values / kde_values.max() * scaling_factor

        # Plot the half violin (left side only, like original)
        ax.fill_betweenx(x_range, positions[i], positions[i] - kde_values,
                        facecolor=colors[i], alpha=0.7, edgecolor='black')

        # Calculate mean and standard deviation
        mean_val = np.mean(group_data)
        std_val = np.std(group_data)
        n_val = len(group_data)
        print(f"{group} {metric} {time_label} mean: {mean_val:.2f}, STD: {std_val:.2f}, SE: {std_val/np.sqrt(n_val):.2f}")

        lower, upper = capped_error(mean_val, std_val, n_val)

        # Plot error bars and means (offset slightly to the right like original)
        ax.errorbar(positions[i] + 0.03, mean_val, yerr=[[lower], [upper]],
                   fmt='o', color=colors[i], capsize=5, markersize=7, elinewidth=2, capthick=2)

        means.append(mean_val)

    # Plot horizontal line from the control group mean through all groups
    ax.plot([positions[0], positions[-1]], [means[0], means[0]], '--', color='black', alpha=0.3, linewidth=2)

    # Add significance brackets if provided
    if brackets:
        for group1_idx, group2_idx, y_pos, height, text in brackets:
            add_significance_bracket(
                ax,
                positions[group1_idx],
                positions[group2_idx],
                y_pos,
                height,
                text
            )

    # Customize the plot
    ax.set_xlim(positions[0] - 0.1, positions[-1] + 0.1)
    ax.set_ylim(0, y_max)
    ax.set_xticks(positions)
    ax.set_xticklabels(names, fontsize=12, rotation=45, ha='right')

    if metric_label is None:
        # Convert metric name to a more readable format
        metric_name = ' '.join(word.capitalize() for word in metric.split('_'))
        y_label = f"{metric_name} (in %)"
    else:
        y_label = metric_label

    ax.set_ylabel(y_label, fontsize=14)
    ax.grid(False)

    # Add grid lines
    yticks = ax.get_yticks()
    x_min = positions[0] - 0.1
    x_max = positions[-1] + 0.1

    for y in yticks:
        ax.hlines(y, xmin=x_min, xmax=x_max, linestyle='--', alpha=0.7, color='lightgray', linewidth=1, zorder=0)

    # Increase y-axis tick size
    ax.tick_params(axis='y', which='major', labelsize=12)

    plt.tight_layout()
    plt.savefig(file_name, format='pdf', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
plot_single_timepoint(df_participants, file_name='../../Results/pilot2/acc_violins_pilot2_real_t1.pdf', metric='real_accuracy', time_label='T1', metric_label='Accuracy (in %)')
plot_single_timepoint(df_participants_follow_up, file_name='../../Results/pilot2/acc_violins_pilot2_real_t2.pdf', metric='real_accuracy', time_label='T2', metric_label='Accuracy (in %)')

In [ ]:
t1_brackets = [
    (0, 1, 100, 1.0, '**'),# Control vs group 2
    (0, 2, 104, 1.0, '***'),# Control vs group 3
    # (0, 4, 112, 1.5, '*'),# Control vs group 4
]

plot_single_timepoint(df_participants, file_name='../../Results/pilot2/acc_violins_pilot2_fake_t1.pdf', metric='fake_accuracy', time_label='T1', metric_label='Accuracy (in %)', brackets=t1_brackets)
plot_single_timepoint(df_participants_follow_up, file_name='../../Results/pilot2/acc_violins_pilot2_fake_t2.pdf', metric='fake_accuracy', time_label='T2', metric_label='Accuracy (in %)')